In [28]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [29]:
load_dotenv()  # Load environment variables from .env file

True

In [30]:
model = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7)

In [31]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    score: float
    feedback: str

In [33]:
class BlogScore(BaseModel):
    """Structured evaluation of a blog post."""
    score: float = Field(description="Quality score from 1 to 10", ge=1, le=10)
    feedback: str = Field(description="Brief justification for the score")


# the model is forced to answer in the shape of BlogScore, so no text parsing is needed.
# gpt-3.5-turbo has no Structured Output API, so ask for function calling explicitly.
scorer = model.with_structured_output(BlogScore, method="function_calling")

In [34]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [35]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [38]:
def score_blog(state: BlogState) -> BlogState:
    content = state['content']

    prompt = f'Score the following blog content on a scale of 1-10 based on quality, clarity, and engagement: \n {content}'

    # returns a BlogScore instance, not an AIMessage - there is no .content here
    result = scorer.invoke(prompt)

    state['score'] = result.score
    state['feedback'] = result.feedback

    return state

In [39]:
graph = StateGraph(BlogState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
graph.add_node("score_blog", score_blog)

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", "score_blog")
graph.add_edge("score_blog", END)

workflow = graph.compile()

In [40]:
workflow_input = {
    "title": "The Future of Artificial Intelligence"
}

workflow_output = workflow.invoke(workflow_input)

In [19]:
workflow_output["content"]

'\n\nI. Introduction\n\nA. Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. This includes learning, reasoning, problem-solving, perception, and even speech recognition. AI has made significant advancements in recent years, with technologies like machine learning, neural networks, and deep learning driving innovation in this field.\n\nB. Currently, AI is being used in various industries such as healthcare, finance, transportation, and education to improve efficiency, accuracy, and decision-making processes.\n\nC. Thesis statement: This blog will delve into the potential future developments and implications of AI, exploring the evolution of AI, potential applications in various sectors, ethical and societal implications, challenges and limitations, and speculations on the future of AI.\n\nII. The Evolution of AI\n\nA. AI has a rich historical background, dating back to the 1950s when the term was first coine

In [41]:
workflow_output

{'title': 'The Future of Artificial Intelligence',
 'outline': "I. Introduction\n    A. Definition of Artificial Intelligence (AI)\n    B. Brief history of AI development\n    C. Importance of AI in today's society\n    \nII. Current Applications of AI\n    A. AI in healthcare\n    B. AI in finance\n    C. AI in transportation\n    D. AI in customer service\n    E. AI in marketing\n    \nIII. Challenges and Limitations of AI\n    A. Ethical concerns\n    B. Bias in AI algorithms\n    C. Job displacement\n    D. Security risks\n    \nIV. Future Trends in AI\n    A. Advancements in deep learning\n    B. Integration of AI with other technologies (e.g. Internet of Things)\n    C. AI in space exploration\n    D. AI in creative fields (e.g. art, music, writing)\n    \nV. Impact of AI on Society\n    A. Economic implications\n    B. Social implications\n    C. Legal implications\n    D. Education and workforce training\n    \nVI. Opportunities for Growth in AI\n    A. Investment in AI researc